## Model Evaluation: Walk-Forward Cross Validation

Before comparing SARIMA forecasts against live World Cup data, the modeel's accuracy will be validated using walk-forward cross-validation. This ensures the baseline forecasts are reliable enough to meaningfully measure World Cup deviations.

**Methodology:**
- Split each city's time series into 2 folds
- Each fold trains on all data before the test window (no data leakage)
- Test window is 12 months per fold
- Metrics: RMSE and MASE averaged across all folds and cities

In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import TimeSeriesSplit 
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.statespace.sarimax import SARIMAX
from pmdarima import auto_arima
import sys
sys.path.append("..")
from constants import target_cities, host_cities, nonhost_cities


In [6]:
# Reading in data
df = pd.read_csv('../data/processed/worldcup_city_seasonal_spending.csv', index_col='Date', parse_dates = True)
print(df.head())

            spend_all  spend_acf     cityname stateabbrev  city_pop2019  \
Date                                                                      
2022-01-31    -0.1300    -0.1580  Los Angeles          CA      10039107   
2022-01-31     0.1810     0.0257      Chicago          IL       5150233   
2022-01-31    -0.0201    -0.0351       Dallas          TX       2635516   
2022-01-31     0.0388     0.0112       Austin          TX       1273954   
2022-01-31    -0.0339    -0.0717    Charlotte          NC       1110356   

            season host_status  
Date                            
2022-01-31  Winter        Host  
2022-01-31  Winter    Non-Host  
2022-01-31  Winter        Host  
2022-01-31  Winter    Non-Host  
2022-01-31  Winter    Non-Host  


In [9]:
# Testing on Atlanta data to ensure pipeline works end-to-end and for debugging purposes

atl_df = df[df['cityname'] == 'Atlanta']
atl_spending = atl_df.spend_acf

# Splitting data into 3 folds

time_split = TimeSeriesSplit(n_splits=2, test_size=12)  # 2 splits with a test size of 12 months each

rmse_scores = [] # store rmse scores to calculate average at the end of loop
mase_scores = [] # store mase scores to calculate average at the end of loop

# Loop through the splits and train/test the model
for train_idx, test_idx in time_split.split(atl_spending):

    atl_train, atl_test = atl_spending.iloc[train_idx], atl_spending.iloc[test_idx]
    
    # Fitting SARIMA model
    auto_model = auto_arima(atl_train, seasonal=True, m=12, seasonal_test='ch', error_action='ignore', suppress_warnings=True)
    sarima_model = SARIMAX(atl_train, order=auto_model.order, seasonal_order=auto_model.seasonal_order, trend='c' if auto_model.with_intercept else None)
    model_fit = sarima_model.fit(disp=False) 
    
    # Generate forecasts from training data
    forecast = model_fit.get_forecast(steps=len(atl_test))
    forecast_mean = forecast.predicted_mean
    forecast_mean.name = "Predicted Forecast" 

    # Computing RMSE (Root Mean Squared Error)
    mse = mean_squared_error(atl_test, forecast_mean)
    rmse = np.sqrt(mse) 
    print(f"RMSE ({atl_test.index[0].strftime('%b %Y')} - {atl_test.index[-1].strftime('%b %Y')}): {rmse:.4f}")
    rmse_scores.append(rmse)

    # MASE (Mean Absolute Scaled Error)
    naive_errors = np.abs(np.diff(atl_train.values))
    mae_naive = np.mean(naive_errors)
    mae_model = np.mean(np.abs(atl_test.values - forecast_mean.values))
    mase = mae_model / mae_naive
    print(f"MASE ({atl_test.index[0].strftime('%b %Y')} - {atl_test.index[-1].strftime('%b %Y')}): {mase:.4f}")
    mase_scores.append(mase)
    print()

RMSE (Jun 2024 - May 2025): 0.0660
MASE (Jun 2024 - May 2025): 1.2632

RMSE (Jun 2025 - May 2026): 0.0729
MASE (Jun 2025 - May 2026): 1.5573



RMSE is used over MSE because it returns error in the original units of the data (percent change), making it directly interpretable. A lower RMSE indicates the model's forecasts are closer to actual spending values.

Atlanta's RMSE was 0.0676 in the Jun 2024–May 2025 window and 0.0729 in Jun 2025–May 2026, showing a slight decline in absolute accuracy in the more recent window rather than an improvement. MASE reinforces this: both folds exceeded 1 (1.26 and 1.56), meaning a naive "no change from last month" forecast would have outperformed SARIMA for Atlanta in both validation windows, despite RMSE remaining low in absolute terms. This pattern, along with a likely explanation, is explored further in the model evaluation notebook.

Add to last notebook for validating model in business context.
These values show that the model had a strong accuracy when predicting Atlanta's spending prices meaning any observed spending above the forecasted baseline can be attributed to World Cup-driven demand rather than normal seasonal variation.

In [ ]:
# Conducting model evaluation across all cities

# Initializing lists to track RMSE and MAPE scores
rmse_scores = []
mase_scores = []

# Initializing lists to track forecast mean and metrics to add to dataset for Tableu visualization
forecast_rows = []
metrics_rows = []

for city in target_cities:
    city_df = df[df['cityname'] == city]
    city_spending = city_df.spend_acf

    # Splitting the data into train/test 

    time_split = TimeSeriesSplit(n_splits=2, test_size=12)

    for fold_num, (train_idx, test_idx) in enumerate(time_split.split(city_spending), start=1):
        city_train, city_test = city_spending.iloc[train_idx], city_spending.iloc[test_idx]

        # Fitting SARIMA model
        auto_model = auto_arima(city_train, seasonal=True, m=12, seasonal_test='ch', error_action='ignore', suppress_warnings=True)
        sarima_model = SARIMAX(city_train, order=auto_model.order, seasonal_order=auto_model.seasonal_order, trend='c' if auto_model.with_intercept else None)
        model_fit = sarima_model.fit(disp=False)

        # Forecast generatoin 
        forecast = model_fit.get_forecast(steps=len(city_test))
        forecast_mean = forecast.predicted_mean

        print(f"{city}")
        # Computing RMSE (Root Mean Squared Error)
        mse = mean_squared_error(city_test, forecast_mean)
        rmse = np.sqrt(mse) 
        print(f"RMSE ({city_test.index[0].strftime('%b %Y')} - {city_test.index[-1].strftime('%b %Y')}): {rmse:.4f}")
        rmse_scores.append(rmse)

        # MASE (Mean Absolute Scaled Error)
        naive_errors = np.abs(np.diff(city_train.values))
        mae_naive = np.mean(naive_errors)
        mae_model = np.mean(np.abs(city_test.values - forecast_mean.values))
        mase = mae_model / mae_naive
        print(f"MASE ({city_test.index[0].strftime('%b %Y')} - {city_test.index[-1].strftime('%b %Y')}): {mase:.4f}")
        mase_scores.append(mase)
        print()

        # Per-month rows for Tableau (actual vs predicted)
        for date, actual_val, predicted_val in zip(city_test.index, city_test.values, forecast_mean.values):
            forecast_rows.append({
                'Date': date,
                'City': city,
                'Fold': fold_num,
                'Actual': actual_val,
                'Predicted': round(predicted_val, 4),
                'Host_status': 'Host' if city in host_cities else 'Non-Host'
            })

        # One row per city/fold summary
        metrics_rows.append({
            'City': city,
            'Fold': fold_num,
            'RMSE': rmse,
            'MASE': mase,
            'Host_status': 'Host' if city in host_cities else 'Non-Host'
        })

# Export both to CSV
forecast_df = pd.DataFrame(forecast_rows)
forecast_df.to_csv('../outputs/Datasets/forecast_vs_actual.csv', index=False)

metrics_df = pd.DataFrame(metrics_rows)
metrics_df.to_csv('../outputs/Datasets/model_metrics_by_city.csv', index=False)

# Looking at evaluation metrics for host, non-host, and all cities
host_rmse = metrics_df[metrics_df['Host_status'] == 'Host']['RMSE'].mean()
host_mase = metrics_df[metrics_df['Host_status'] == 'Host']['MASE'].mean()
nonhost_rmse = metrics_df[metrics_df['Host_status'] == 'Non-Host']['RMSE'].mean()
nonhost_mase = metrics_df[metrics_df['Host_status'] == 'Non-Host']['MASE'].mean()
overall_rmse = metrics_df['RMSE'].mean()
overall_mase = metrics_df['MASE'].mean()

print(f"Host cities average RMSE: {host_rmse:.4f}")
print(f"Host cities average MASE: {host_mase:.4f}")
print(f"Non-Host cities average RMSE: {nonhost_rmse:.4f}")
print(f"Non-host cities average MASE: {nonhost_mase:.4f}")
print(f"All cities average RMSE: {overall_rmse:.4f}")
print(f"All cities average MASE: {overall_mase:.4f}")

    

Los Angeles
RMSE (Jun 2024 - May 2025): 0.0195
MASE (Jun 2024 - May 2025): 0.6172

Los Angeles
RMSE (Jun 2025 - May 2026): 0.0165
MASE (Jun 2025 - May 2026): 0.5387

Denver
RMSE (Jun 2024 - May 2025): 0.0245
MASE (Jun 2024 - May 2025): 0.5197

Denver
RMSE (Jun 2025 - May 2026): 0.0290
MASE (Jun 2025 - May 2026): 0.8094

Dallas
RMSE (Jun 2024 - May 2025): 0.0314
MASE (Jun 2024 - May 2025): 0.5468

Dallas
RMSE (Jun 2025 - May 2026): 0.0320
MASE (Jun 2025 - May 2026): 0.6661

Chicago
RMSE (Jun 2024 - May 2025): 0.0456
MASE (Jun 2024 - May 2025): 0.7810

Chicago
RMSE (Jun 2025 - May 2026): 0.0708
MASE (Jun 2025 - May 2026): 1.4868

Boston
RMSE (Jun 2024 - May 2025): 0.0769
MASE (Jun 2024 - May 2025): 1.0476

Boston
RMSE (Jun 2025 - May 2026): 0.0530
MASE (Jun 2025 - May 2026): 0.6804

Atlanta
RMSE (Jun 2024 - May 2025): 0.0660
MASE (Jun 2024 - May 2025): 1.2632

Atlanta
RMSE (Jun 2025 - May 2026): 0.0729
MASE (Jun 2025 - May 2026): 1.5573

Kansas City
RMSE (Jun 2024 - May 2025): 0.0827
MAS

MAPE was initially used to evaluate percentage error but was found to be unreliable for this dataset, several cities have test-period actual values very close to zero, causing MAPE to inflate to values exceeding 900% even when the model's absolute error remained small. MASE was used instead as it compares model error to a naive "no change" baseline forecast rather than dividing by the actual value directly. 

Twelve of the 18 city/fold combinations had a MASE under 1, and a few stood out as genuinely strong: Austin (0.26), Denver (0.52), and Los Angeles (around 0.55) all beat a naive "assume no change" guess, suggesting these cities have real seasonal patterns the model could latch onto. The other six combinations, including both Atlanta folds, Kansas City's second fold, Chicago's second fold, and single folds for Boston and Austin, came in above 1. That doesn't mean the model is broken; RMSE stayed low and consistent everywhere (0.05 to 0.09), so the actual size of the errors was small. It just means that for these particular city/fold windows guessing "nothing changed since last month" would have done slightly better than SARIMA. Part of this traces back to something we found earlier: three of the six weaker cases (Atlanta, Kansas City, Chicago) show a sharp dip-then-spike right before the test window starts, which likely threw off the fitted model. Given that, I'm treating each city's forecast with confidence roughly matching its own MASE score: leaning on Austin and Denver's numbers, and reading Atlanta and Kansas City's forecasts with a bit more skepticism when comparing them to actual World Cup spending later.

Averaged across cities, host cities showed both higher RMSE (0.0540 vs. 0.0376) and higher MASE (0.94 vs. 0.79) than non-host cities, indicating the model's baseline forecasts are meaningfully less reliable for host cities as a group. This should be factored into interpretation of any observed divergence during the actual tournament period, as host cities carry inherently more forecast uncertainty independent of World Cup effects.